In [10]:
import MetaTrader5 as mt5
import json
from datetime import datetime, timedelta

def obter_dados_moeda_com_spread_eficiente(data_inicio, data_fim, intervalo_dias=30):
    # Inicializar a conexão com o MetaTrader 5
    if not mt5.initialize():
        print("Erro ao inicializar o MetaTrader 5")
        mt5.shutdown()
        return

    # Definir o símbolo
    symbol = "GOLD"
    CONTRATO = 100
    # Definir a lista para salvar todos os dados coletados
    todos_dados = []
    intervalo = timedelta(days=intervalo_dias)  # Divisão em intervalos de dias
    data_atual = data_inicio

    while data_atual < data_fim:
        # Definir o final do intervalo
        data_intervalo_fim = min(data_atual + intervalo, data_fim)

        print(f"Buscando dados de {data_atual} até {data_intervalo_fim}")

        # Obter os dados de candle para o intervalo atual
        rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_M1, data_atual, data_intervalo_fim)
        
        if rates is None or len(rates) == 0:
            print(f"Erro ao obter os dados de candle para {symbol} no intervalo de {data_atual} até {data_intervalo_fim}.")
        else:
            # Buscar os ticks para todo o intervalo de uma vez
            ticks = mt5.copy_ticks_range(symbol, data_atual, data_intervalo_fim, mt5.COPY_TICKS_ALL)

            if ticks is not None and len(ticks) > 0:
                # Criar dicionário de spreads calculados para cada minuto
                tick_spreads = {}
                for tick in ticks:
                    tick_time = datetime.fromtimestamp(tick['time']).strftime('%Y.%m.%d %H:%M')
                    if tick_time not in tick_spreads:
                        tick_spreads[tick_time] = {'ask': [], 'bid': []}
                    tick_spreads[tick_time]['ask'].append(tick['ask'])
                    tick_spreads[tick_time]['bid'].append(tick['bid'])

                # Calcular o spread médio por minuto
                for tick_time, prices in tick_spreads.items():
                    avg_ask = sum(prices['ask']) / len(prices['ask'])
                    avg_bid = sum(prices['bid']) / len(prices['bid'])
                    tick_spreads[tick_time] = avg_ask - avg_bid
            else:
                tick_spreads = {}

            for rate in rates:
                # Coletar as informações básicas de cada candle
                time = datetime.fromtimestamp(rate['time']).strftime('%Y.%m.%d %H:%M:%S')
                open_price = float(rate['open'])
                high = float(rate['high'])
                low = float(rate['low'])
                close = float(rate['close'])
                tick_volume = int(rate['tick_volume'])

                # Usar o spread dos ticks, se disponível, ou o spread padrão
                minute_key = datetime.fromtimestamp(rate['time']).strftime('%Y.%m.%d %H:%M')
                spread = CONTRATO*tick_spreads.get(minute_key, high - low)  # Usar o spread dos ticks ou o padrão

                # Adicionar os dados processados à lista
                todos_dados.append({
                    "TIME": time,
                    "OPEN": open_price,
                    "HIGH": high,
                    "LOW": low,
                    "CLOSE": close,
                    "TICK_VOLUME": tick_volume,
                    "SPREAD": spread
                })

        # Avançar para o próximo intervalo
        data_atual = data_intervalo_fim

    # Salvar os dados em um arquivo JSON
    nome_do_arquivo = f"[HISTORICO]_[{symbol}]_{data_inicio.strftime('%Y%m%d')}_{data_fim.strftime('%Y%m%d')}.json"
    with open(nome_do_arquivo, "w") as arquivo:
        json.dump(todos_dados, arquivo, indent=4)

    print(f"\nDados salvos em {nome_do_arquivo}")

    # Finalizar a conexão com o MetaTrader 5
    mt5.shutdown()

# Definir as datas de início e fim
data_inicio = datetime(2020, 1, 1, 0, 0)  # Exemplo de data de início
data_fim = datetime(2024, 12, 31, 0, 0)     # Exemplo de data de fim

# Chamar a função para obter os dados e salvar no arquivo
obter_dados_moeda_com_spread_eficiente(data_inicio, data_fim, intervalo_dias=30)  # Busca em intervalos de 30 dias


Buscando dados de 2020-01-01 00:00:00 até 2020-01-31 00:00:00
Buscando dados de 2020-01-31 00:00:00 até 2020-03-01 00:00:00
Buscando dados de 2020-03-01 00:00:00 até 2020-03-31 00:00:00
Buscando dados de 2020-03-31 00:00:00 até 2020-04-30 00:00:00
Buscando dados de 2020-04-30 00:00:00 até 2020-05-30 00:00:00
Buscando dados de 2020-05-30 00:00:00 até 2020-06-29 00:00:00
Buscando dados de 2020-06-29 00:00:00 até 2020-07-29 00:00:00
Buscando dados de 2020-07-29 00:00:00 até 2020-08-28 00:00:00
Buscando dados de 2020-08-28 00:00:00 até 2020-09-27 00:00:00
Buscando dados de 2020-09-27 00:00:00 até 2020-10-27 00:00:00
Buscando dados de 2020-10-27 00:00:00 até 2020-11-26 00:00:00
Buscando dados de 2020-11-26 00:00:00 até 2020-12-26 00:00:00
Buscando dados de 2020-12-26 00:00:00 até 2021-01-25 00:00:00
Buscando dados de 2021-01-25 00:00:00 até 2021-02-24 00:00:00
Buscando dados de 2021-02-24 00:00:00 até 2021-03-26 00:00:00
Buscando dados de 2021-03-26 00:00:00 até 2021-04-25 00:00:00
Buscando